# Adaptive Observation Gain Control via RL — NeuroSim XPBD
**BU.520.750 · AI-Driven Sequential Decision Making · Spring II 2026**

**Runtime:** `Runtime → Change runtime type → T4 GPU`  
**Order:** Run all cells top-to-bottom. Training ~15-25 min on T4.

---
| Section | What it does |
|---|---|
| 1 | Install deps + GPU check |
| 2 | Paste simulator code (SkinXPBD_Pure) |
| 3 | RL environment (Gymnasium wrapper) |
| 4 | Q-Learning + SARSA agents |
| 5 | Baselines |
| 6 | Training |
| 7 | Evaluation |
| 8 | All figures |

## 1 · Setup

In [1]:
# Install / verify dependencies
!pip install -q scipy matplotlib numpy

import torch, sys, os
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {DEVICE}')

os.makedirs('rl_results', exist_ok=True)

Python  : 3.12.13
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-40GB
VRAM    : 42.4 GB

Using device: cuda


## 2 · XPBD Simulator (SkinXPBD_Pure)
Pure-PyTorch three-layer tissue simulator — no Newton/Warp dependency needed on Colab.

In [2]:
import time
import numpy as np
import torch

# ── Layer constants (Li et al. 2022 OCE) ──────────────────────────────────
LAYER_HYPO = 0   # Hypodermis — softest, deepest
LAYER_DERM = 1   # Dermis     — primary load-bearing
LAYER_EPI  = 2   # Epidermis  — stiffest surface layer

LAYER_KE  = {LAYER_HYPO: 10.0,  LAYER_DERM: 48.0,  LAYER_EPI: 80.0}   # N/m
LAYER_KD  = {LAYER_HYPO: 0.15,  LAYER_DERM: 0.08,  LAYER_EPI: 0.05}
INTER_KE  = {(LAYER_HYPO, LAYER_DERM): 30.0,
             (LAYER_DERM, LAYER_EPI):  60.0}


def build_skin_mesh(rows=30, cols=30, spacing=0.005):
    N_per = rows * cols
    N     = 3 * N_per
    pos   = np.zeros((N, 3), dtype=np.float32)
    lids  = np.zeros(N,      dtype=np.int32)
    layer_y = {LAYER_HYPO: 0.0, LAYER_DERM: spacing, LAYER_EPI: 2.0 * spacing}

    for lyr in (LAYER_HYPO, LAYER_DERM, LAYER_EPI):
        base = lyr * N_per
        y    = layer_y[lyr]
        for r in range(rows):
            for c in range(cols):
                i       = base + r * cols + c
                pos[i]  = [(c - cols/2.0)*spacing, y, (r - rows/2.0)*spacing]
                lids[i] = lyr

    edges_list, ke_list, kd_list = [], [], []
    def add(i, j, ke, kd):
        edges_list.append([i, j]); ke_list.append(ke); kd_list.append(kd)

    for lyr in (LAYER_HYPO, LAYER_DERM, LAYER_EPI):
        base = lyr * N_per
        ke, kd = LAYER_KE[lyr], LAYER_KD[lyr]
        for r in range(rows):
            for c in range(cols):
                i = base + r * cols + c
                if c+1 < cols:              add(i, base+r*cols+c+1,      ke, kd)
                if r+1 < rows:              add(i, base+(r+1)*cols+c,    ke, kd)
                if r+1<rows and c+1<cols:   add(i, base+(r+1)*cols+c+1,  ke, kd)
                if r+1<rows and c-1>=0:     add(i, base+(r+1)*cols+c-1,  ke, kd)

    for (lo, hi) in ((LAYER_HYPO, LAYER_DERM), (LAYER_DERM, LAYER_EPI)):
        ke = INTER_KE[(lo, hi)]
        for r in range(rows):
            for c in range(cols):
                add(lo*N_per+r*cols+c, hi*N_per+r*cols+c, ke, 0.08)

    edges  = np.array(edges_list,  dtype=np.int32)
    ke_arr = np.array(ke_list,     dtype=np.float32)
    kd_arr = np.array(kd_list,     dtype=np.float32)
    print(f'[SkinMesh] {rows}×{cols} × 3 layers = {N} particles, {len(edges)} springs')
    return pos, edges, lids, ke_arr, kd_arr


class SkinXPBD_Pure:
    def __init__(self, rows=30, cols=30, spacing=0.005,
                 substeps=10, n_iter=20, gravity_y=0.0,
                 damping=0.98, dt=1.0/60.0, device='cuda'):
        self.dt, self.substeps, self.n_iter = dt, substeps, n_iter
        self.gravity_y, self.damping = gravity_y, damping
        self._dev  = device
        self.rows, self.cols = rows, cols
        self.spacing = spacing

        pos, edges, lids, ke_arr, kd_arr = build_skin_mesh(rows, cols, spacing)
        N = len(pos); E = len(edges)
        self._N = N
        self.layer_ids    = lids
        self.surface_mask = (lids == LAYER_EPI)
        self.rest_np      = pos.copy()

        self.pos      = torch.tensor(pos,  dtype=torch.float32, device=device)
        self.vel      = torch.zeros(N, 3,  dtype=torch.float32, device=device)
        self.inv_mass = torch.ones(N,      dtype=torch.float32, device=device)

        self.ei       = torch.tensor(edges[:,0], dtype=torch.long,  device=device)
        self.ej       = torch.tensor(edges[:,1], dtype=torch.long,  device=device)
        pi = pos[edges[:,0]]; pj = pos[edges[:,1]]
        self.rest_len = torch.tensor(
            np.linalg.norm(pi-pj, axis=1), dtype=torch.float32, device=device)
        self.alpha    = 1.0 / torch.tensor(ke_arr, dtype=torch.float32, device=device)
        self.kd_t     = torch.tensor(kd_arr, dtype=torch.float32, device=device)

        deg = torch.zeros(N, dtype=torch.float32, device=device)
        deg.scatter_add_(0, self.ei, torch.ones(E, dtype=torch.float32, device=device))
        deg.scatter_add_(0, self.ej, torch.ones(E, dtype=torch.float32, device=device))
        self.degree = deg.clamp(min=1.0)

        print(f'[SkinXPBD_Pure] N={N}, surface={int(self.surface_mask.sum())}')
        print(f'  gravity={gravity_y:.1f} m/s²  substeps={substeps}  n_iter={n_iter}')

    def apply_tool(self, depth_m, surf_idx, tool_weights):
        if depth_m < 1e-8: return
        rest_y  = torch.tensor(self.rest_np[surf_idx, 1], dtype=torch.float32, device=self._dev)
        weights = torch.tensor(tool_weights,              dtype=torch.float32, device=self._dev)
        target  = rest_y - depth_m * weights
        cur_y   = self.pos[surf_idx, 1]
        push    = cur_y > target
        if push.any():
            idx = torch.tensor(surf_idx, dtype=torch.long, device=self._dev)[push]
            self.pos[idx, 1] = target[push]

    def step(self):
        t0     = time.perf_counter()
        sub_dt = self.dt / self.substeps
        alpha_t = self.alpha / (sub_dt ** 2)

        for _ in range(self.substeps):
            if abs(self.gravity_y) > 1e-9:
                self.vel[:, 1] += sub_dt * self.gravity_y
            pos_pred = self.pos + sub_dt * self.vel
            lam = torch.zeros(len(self.ei), dtype=torch.float32, device=self._dev)

            for _ in range(self.n_iter):
                pi   = pos_pred[self.ei]; pj = pos_pred[self.ej]
                diff = pi - pj
                dist = torch.norm(diff, dim=1).clamp(min=1e-9)
                n_hat = diff / dist.unsqueeze(1)
                C    = dist - self.rest_len
                wi   = self.inv_mass[self.ei]; wj = self.inv_mass[self.ej]
                dlam = -(C + alpha_t * lam) / (wi + wj + alpha_t)
                lam  = lam + dlam
                ci   = (wi * dlam / self.degree[self.ei]).unsqueeze(1) * n_hat
                cj   = (wj * dlam / self.degree[self.ej]).unsqueeze(1) * n_hat
                dx   = torch.zeros_like(pos_pred)
                dx.scatter_add_(0, self.ei.unsqueeze(1).expand_as(ci),  ci)
                dx.scatter_add_(0, self.ej.unsqueeze(1).expand_as(cj), -cj)
                pos_pred = pos_pred + dx

            self.vel  = (pos_pred - self.pos) / sub_dt
            self.vel  = self.vel * self.damping
            self.pos  = pos_pred

        stress = torch.norm(self.vel, dim=1)
        return self.pos.clone(), stress.clone(), (time.perf_counter()-t0)*1000

    @property
    def n_particles(self): return self._N

print('✓ Simulator defined')

✓ Simulator defined


In [3]:
# Quick sanity check: mesh should not drift with gravity=0
print('Sanity check: gravity=0 → no drift')
_sim = SkinXPBD_Pure(rows=10, cols=10, spacing=0.005,
                     substeps=5, n_iter=5, gravity_y=0.0,
                     damping=0.98, device=DEVICE)
for k in range(3):
    pos, _, ms = _sim.step()
    drift = float((pos.cpu() - torch.tensor(_sim.rest_np)).abs().max()) * 1000
    print(f'  step {k}: max drift = {drift:.6f} mm  ({ms:.1f} ms)')
del _sim
print('✓ Simulator verified')

Sanity check: gravity=0 → no drift
[SkinMesh] 10×10 × 3 layers = 300 particles, 1226 springs
[SkinXPBD_Pure] N=300, surface=100
  gravity=0.0 m/s²  substeps=5  n_iter=5
  step 0: max drift = 0.000000 mm  (244.4 ms)
  step 1: max drift = 0.000000 mm  (11.5 ms)
  step 2: max drift = 0.000000 mm  (10.8 ms)
✓ Simulator verified


## 3 · RL Environment

In [4]:
from scipy.spatial import KDTree
from typing import Optional, Tuple, Dict, List

# ── MDP Constants ─────────────────────────────────────────────────────────
K_OBS_MIN    = 5.0
K_OBS_MAX    = 200.0
K_OBS_INIT   = 48.0       # dermis stiffness — theoretical starting point
K_OBS_STEP   = 0.05       # 5% multiplicative step
LAMBDA_REG   = 0.01       # gain-stability penalty weight
NOISE_SIGMA  = 0.0015     # 1.5 mm depth sensor noise
EPISODE_LEN  = 300        # frames per episode (5 s @ 60 Hz)
N_STATE_BINS = 5
N_ACTIONS    = 3          # {decrease, hold, increase}
N_STATES     = N_STATE_BINS ** 4   # 625

EP_NORMAL    = 0
EP_OCCLUSION = 1
EP_CONTACT   = 2
EPISODE_TYPES = [EP_NORMAL, EP_OCCLUSION, EP_CONTACT]
EP_NAMES      = ['Normal', 'Occlusion', 'Contact']

STATE_LOW  = np.array([0.0,  0.0,   0.0,  K_OBS_MIN], dtype=np.float32)
STATE_HIGH = np.array([15.0, 0.05,  5.0,  K_OBS_MAX], dtype=np.float32)


def chamfer_distance(pred, obs):
    if len(obs) == 0 or len(pred) == 0: return 0.0
    d_p2o, _ = KDTree(obs).query(pred)
    d_o2p, _ = KDTree(pred).query(obs)
    return 0.5 * (d_p2o.mean() + d_o2p.mean())


def discretise(obs):
    clipped = np.clip(obs, STATE_LOW, STATE_HIGH)
    normed  = (clipped - STATE_LOW) / (STATE_HIGH - STATE_LOW + 1e-9)
    bins    = (normed * N_STATE_BINS).astype(int).clip(0, N_STATE_BINS-1)
    idx = 0
    for b in bins: idx = idx * N_STATE_BINS + int(b)
    return idx


class SkinTrackingEnv:
    def __init__(self, rows=30, cols=30, spacing=0.005,
                 substeps=10, n_iter=20, device='cuda',
                 episode_type=None, seed=42):
        self.rng = np.random.default_rng(seed)
        self._ep_type_fixed = episode_type
        self.device = device

        print('[Env] Building simulator ...')
        self.sim = SkinXPBD_Pure(
            rows=rows, cols=cols, spacing=spacing,
            substeps=substeps, n_iter=n_iter,
            gravity_y=0.0, damping=0.98, dt=1.0/60.0, device=device)

        self._rest_pos = self.sim.rest_np.copy()
        self._surf_idx = np.where(self.sim.surface_mask)[0]

        surf_rest = self._rest_pos[self._surf_idx]
        xz_dist   = np.sqrt(surf_rest[:,0]**2 + surf_rest[:,2]**2)
        self._tool_weights = np.exp(-0.5 * (xz_dist / 0.007)**2)
        self._press_depth  = 0.008

        self.k_obs = K_OBS_INIT
        self.step_num = 0
        self.ep_type  = EP_NORMAL
        self._prev_cloud = None
        print('[Env] Ready.')

    def reset(self, episode_type=None):
        rest_t = torch.tensor(self._rest_pos, dtype=torch.float32, device=self.device)
        self.sim.pos = rest_t.clone()
        self.sim.vel = torch.zeros_like(self.sim.vel)
        if episode_type is not None:          self.ep_type = episode_type
        elif self._ep_type_fixed is not None: self.ep_type = self._ep_type_fixed
        else: self.ep_type = int(self.rng.integers(0, 3))
        self.k_obs = K_OBS_INIT
        self.step_num = 0
        self._prev_cloud = None
        obs, _, _ = self._compute_obs(np.zeros((len(self._surf_idx), 3)))
        return obs

    def step(self, action):
        prev_k = self.k_obs
        if   action == 0: self.k_obs = max(K_OBS_MIN, self.k_obs*(1-K_OBS_STEP))
        elif action == 2: self.k_obs = min(K_OBS_MAX, self.k_obs*(1+K_OBS_STEP))
        dk = abs(self.k_obs - prev_k)

        gt_surf = self._rest_pos[self._surf_idx].copy()
        noise   = self.rng.normal(0, NOISE_SIGMA, gt_surf.shape).astype(np.float32)
        cloud   = gt_surf + noise

        if self.ep_type == EP_OCCLUSION and 100 <= self.step_num <= 130:
            n_zero = int(0.4 * len(cloud))
            cloud[self.rng.choice(len(cloud), n_zero, replace=False)] = 0.0

        if self.ep_type == EP_CONTACT and self.step_num == 150:
            self.sim.apply_tool(self._press_depth, self._surf_idx, self._tool_weights)

        self._inject_obs_springs(cloud)
        pos, stress, _ = self.sim.step()
        obs, d_cd, delta_cloud = self._compute_obs(cloud)

        reward = -d_cd * 1000.0 - LAMBDA_REG * dk
        self.step_num += 1
        self._prev_cloud = cloud.copy()
        done = (self.step_num >= EPISODE_LEN)
        info = {'d_cd_mm': d_cd*1000, 'k_obs': self.k_obs,
                'ep_type': self.ep_type, 'step': self.step_num,
                'mean_speed': float(stress.mean().item())}
        return obs, reward, done, info

    def _inject_obs_springs(self, cloud):
        delta_max = 0.040
        surf_pos  = self.sim.pos[self._surf_idx].cpu().numpy()
        valid     = cloud[np.any(cloud != 0, axis=1)]
        if len(valid) == 0: return
        _, nn = KDTree(valid).query(surf_pos)
        delta = valid[nn] - surf_pos
        mags  = np.linalg.norm(delta, axis=1, keepdims=True).clip(min=1e-9)
        f_obs = (self.k_obs * delta * np.minimum(1.0, delta_max/mags)).astype(np.float32)
        dv    = torch.tensor(f_obs * self.sim.dt, dtype=torch.float32, device=self.device)
        self.sim.vel[self._surf_idx] += dv

    def _compute_obs(self, cloud):
        surf_pos   = self.sim.pos[self._surf_idx].cpu().numpy()
        mean_speed = float(torch.norm(self.sim.vel, dim=1).mean().item())
        valid      = cloud[np.any(cloud != 0, axis=1)]
        d_cd       = chamfer_distance(surf_pos, valid) if len(valid) > 0 else 0.0
        if self._prev_cloud is not None:
            pv = self._prev_cloud[np.any(self._prev_cloud != 0, axis=1)]
            delta_cloud = float(np.linalg.norm(
                valid.mean(0)-pv.mean(0)))*1000 if (len(pv)>0 and len(valid)>0) else 0.0
        else:
            delta_cloud = 0.0
        obs = np.array([d_cd*1000, mean_speed, delta_cloud, self.k_obs], dtype=np.float32)
        return obs, d_cd, delta_cloud

print('✓ Environment defined')

✓ Environment defined


## 4 · Q-Learning & SARSA Agents

In [5]:
import random, pickle

class QLearningAgent:
    """Off-policy tabular Q-learning with ε-greedy exploration.
    Q(s,a) ← Q(s,a) + α[r + γ max_a' Q(s',a') − Q(s,a)]
    """
    def __init__(self, alpha=0.3, gamma=0.95,
                 eps_start=1.0, eps_end=0.05, eps_decay=500):
        self.alpha, self.gamma = alpha, gamma
        self.eps_start, self.eps_end, self.eps_decay = eps_start, eps_end, eps_decay
        self.episode = 0
        self.Q = np.zeros((N_STATES, N_ACTIONS), dtype=np.float64)

    @property
    def epsilon(self):
        frac = min(1.0, self.episode / max(self.eps_decay, 1))
        return self.eps_start + frac*(self.eps_end - self.eps_start)

    def select_action(self, state, greedy=False):
        if not greedy and random.random() < self.epsilon:
            return random.randint(0, N_ACTIONS-1)
        return int(np.argmax(self.Q[state]))

    def update(self, s, a, r, s_next, done):
        target = r if done else r + self.gamma * np.max(self.Q[s_next])
        self.Q[s, a] += self.alpha * (target - self.Q[s, a])

    def end_episode(self): self.episode += 1

    def save(self, path):
        with open(path,'wb') as f: pickle.dump({'Q':self.Q,'ep':self.episode},f)


class SARSAAgent:
    """On-policy SARSA.
    Q(s,a) ← Q(s,a) + α[r + γ Q(s',a') − Q(s,a)]
    """
    def __init__(self, alpha=0.3, gamma=0.95,
                 eps_start=1.0, eps_end=0.05, eps_decay=500):
        self.alpha, self.gamma = alpha, gamma
        self.eps_start, self.eps_end, self.eps_decay = eps_start, eps_end, eps_decay
        self.episode = 0
        self.Q = np.zeros((N_STATES, N_ACTIONS), dtype=np.float64)

    @property
    def epsilon(self):
        frac = min(1.0, self.episode / max(self.eps_decay, 1))
        return self.eps_start + frac*(self.eps_end - self.eps_start)

    def select_action(self, state, greedy=False):
        if not greedy and random.random() < self.epsilon:
            return random.randint(0, N_ACTIONS-1)
        return int(np.argmax(self.Q[state]))

    def update(self, s, a, r, s_next, a_next, done):
        target = r if done else r + self.gamma * self.Q[s_next, a_next]
        self.Q[s, a] += self.alpha * (target - self.Q[s, a])

    def end_episode(self): self.episode += 1

    def save(self, path):
        with open(path,'wb') as f: pickle.dump({'Q':self.Q,'ep':self.episode},f)


print('✓ Agents defined')

✓ Agents defined


## 5 · Baselines & Episode Runners

In [6]:
class FixedGainBaseline:
    def __init__(self, k_obs, env):
        self.k_obs = k_obs; self.env = env
    def reset(self): self.env.k_obs = self.k_obs
    def select_action(self, s): return 1   # always hold

class RandomBaseline:
    def select_action(self, s): return random.randint(0, N_ACTIONS-1)


def run_episode_ql(env, agent, ep_type=None, train=True):
    obs   = env.reset(episode_type=ep_type)
    state = discretise(obs)
    total_r, total_cd, steps, k_traj = 0.0, 0.0, 0, []
    while True:
        action = agent.select_action(state, greedy=not train)
        obs_next, reward, done, info = env.step(action)
        s_next = discretise(obs_next)
        if train: agent.update(state, action, reward, s_next, done)
        total_r += reward; total_cd += info['d_cd_mm']
        k_traj.append(info['k_obs'])
        state = s_next; steps += 1
        if done: break
    if train: agent.end_episode()
    return {'total_reward': total_r, 'mean_cd_mm': total_cd/steps,
            'k_traj': k_traj, 'ep_type': info['ep_type']}


def run_episode_sarsa(env, agent, ep_type=None, train=True):
    obs    = env.reset(episode_type=ep_type)
    state  = discretise(obs)
    action = agent.select_action(state)
    total_r, total_cd, steps, k_traj = 0.0, 0.0, 0, []
    while True:
        obs_next, reward, done, info = env.step(action)
        s_next = discretise(obs_next)
        a_next = agent.select_action(s_next)
        if train: agent.update(state, action, reward, s_next, a_next, done)
        total_r += reward; total_cd += info['d_cd_mm']
        k_traj.append(info['k_obs'])
        state, action = s_next, a_next; steps += 1
        if done: break
    if train: agent.end_episode()
    return {'total_reward': total_r, 'mean_cd_mm': total_cd/steps,
            'k_traj': k_traj, 'ep_type': info['ep_type']}


def run_episode_baseline(env, baseline, ep_type=None):
    obs   = env.reset(episode_type=ep_type)
    if hasattr(baseline, 'reset'): baseline.reset()
    state = discretise(obs)
    total_r, total_cd, steps, k_traj = 0.0, 0.0, 0, []
    while True:
        action = baseline.select_action(state)
        obs_next, reward, done, info = env.step(action)
        state = discretise(obs_next)
        total_r += reward; total_cd += info['d_cd_mm']
        k_traj.append(info['k_obs'])
        steps += 1
        if done: break
    return {'total_reward': total_r, 'mean_cd_mm': total_cd/steps,
            'k_traj': k_traj, 'ep_type': info['ep_type']}


print('✓ Baselines & runners defined')

✓ Baselines & runners defined


## 6 · Build Environment & Train
⏱ **~15–25 min on T4** for 1000 episodes each. Reduce `N_TRAIN` to 200 for a quick smoke-test.

In [7]:
# ── Config ────────────────────────────────────────────────────────────────
N_TRAIN   = 1000   # ← set to 200 for a quick 3-min test run
N_TEST    = 20     # evaluation episodes per condition
EVAL_EVERY = 50
ALPHA     = 0.3
GAMMA     = 0.95
EPS_DECAY = 500

random.seed(42); np.random.seed(42)

env = SkinTrackingEnv(rows=30, cols=30, spacing=0.005,
                      substeps=10, n_iter=20, device=DEVICE, seed=42)

ql    = QLearningAgent(alpha=ALPHA, gamma=GAMMA,
                        eps_start=1.0, eps_end=0.05, eps_decay=EPS_DECAY)
sarsa = SARSAAgent(alpha=ALPHA, gamma=GAMMA,
                    eps_start=1.0, eps_end=0.05, eps_decay=EPS_DECAY)
print(f'\n✓ Environment + agents ready | Training for {N_TRAIN} episodes each')

[Env] Building simulator ...
[SkinMesh] 30×30 × 3 layers = 2700 particles, 12066 springs
[SkinXPBD_Pure] N=2700, surface=900
  gravity=0.0 m/s²  substeps=10  n_iter=20
[Env] Ready.

✓ Environment + agents ready | Training for 1000 episodes each


In [ ]:
# ── Train Q-Learning ──────────────────────────────────────────────────────
print('═'*55)
print('  Training Q-Learning')
print('═'*55)

ql_hist = {'train_reward':[], 'train_cd':[], 'eval_cd':[], 'eval_episodes':[], 'epsilon':[]}

t0 = time.time()
for ep in range(N_TRAIN):
    res = run_episode_ql(env, ql, train=True)
    ql_hist['train_reward'].append(res['total_reward'])
    ql_hist['train_cd'].append(res['mean_cd_mm'])
    ql_hist['epsilon'].append(ql.epsilon)

    if (ep+1) % EVAL_EVERY == 0:
        eval_cds = []
        for et in EPISODE_TYPES:
            r = run_episode_ql(env, ql, ep_type=et, train=False)
            eval_cds.append(r['mean_cd_mm'])
        ql_hist['eval_cd'].append(np.mean(eval_cds))
        ql_hist['eval_episodes'].append(ep+1)
        print(f'  ep={ep+1:4d}  ε={ql.epsilon:.3f}  '
              f'train_cd={np.mean(ql_hist["train_cd"][-EVAL_EVERY:]):.3f}mm  '
              f'eval_cd={ql_hist["eval_cd"][-1]:.3f}mm  '
              f'({time.time()-t0:.0f}s)')

ql.save('rl_results/ql_agent.pkl')
print(f'\n✓ Q-Learning done in {time.time()-t0:.0f}s')

═══════════════════════════════════════════════════════
  Training Q-Learning
═══════════════════════════════════════════════════════


In [ ]:
# ── Train SARSA ───────────────────────────────────────────────────────────
print('═'*55)
print('  Training SARSA')
print('═'*55)

sarsa_hist = {'train_reward':[], 'train_cd':[], 'eval_cd':[], 'eval_episodes':[], 'epsilon':[]}

t0 = time.time()
for ep in range(N_TRAIN):
    res = run_episode_sarsa(env, sarsa, train=True)
    sarsa_hist['train_reward'].append(res['total_reward'])
    sarsa_hist['train_cd'].append(res['mean_cd_mm'])
    sarsa_hist['epsilon'].append(sarsa.epsilon)

    if (ep+1) % EVAL_EVERY == 0:
        eval_cds = []
        for et in EPISODE_TYPES:
            r = run_episode_sarsa(env, sarsa, ep_type=et, train=False)
            eval_cds.append(r['mean_cd_mm'])
        sarsa_hist['eval_cd'].append(np.mean(eval_cds))
        sarsa_hist['eval_episodes'].append(ep+1)
        print(f'  ep={ep+1:4d}  ε={sarsa.epsilon:.3f}  '
              f'train_cd={np.mean(sarsa_hist["train_cd"][-EVAL_EVERY:]):.3f}mm  '
              f'eval_cd={sarsa_hist["eval_cd"][-1]:.3f}mm  '
              f'({time.time()-t0:.0f}s)')

sarsa.save('rl_results/sarsa_agent.pkl')
print(f'\n✓ SARSA done in {time.time()-t0:.0f}s')

## 7 · Evaluation

In [ ]:
baselines = {
    'Fixed 48 N/m':  FixedGainBaseline(48.0, env),
    'Fixed 80 N/m':  FixedGainBaseline(80.0, env),
    'Fixed 10 N/m':  FixedGainBaseline(10.0, env),
    'Random Policy': RandomBaseline(),
}

all_agents = {'Q-Learning': ql, 'SARSA': sarsa, **baselines}
results    = {name: {et: [] for et in EP_NAMES} for name in all_agents}

print('Running evaluation ...')
for name, agent_or_base in all_agents.items():
    for et_code, et_name in zip(EPISODE_TYPES, EP_NAMES):
        for _ in range(N_TEST):
            if isinstance(agent_or_base, QLearningAgent):
                r = run_episode_ql(env, agent_or_base, ep_type=et_code, train=False)
            elif isinstance(agent_or_base, SARSAAgent):
                r = run_episode_sarsa(env, agent_or_base, ep_type=et_code, train=False)
            else:
                r = run_episode_baseline(env, agent_or_base, ep_type=et_code)
            results[name][et_name].append(r['mean_cd_mm'])

print('\n' + '='*65)
print(f'{"Agent":24s}  {"Normal":>12s}  {"Occlusion":>12s}  {"Contact":>12s}')
print('-'*65)
for name, res in results.items():
    row = f'{name:24s}'
    for et in EP_NAMES:
        m, s = np.mean(res[et]), np.std(res[et])
        row += f'  {m:5.3f}±{s:.3f}  '
    print(row)
print('='*65)

## 8 · Figures

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

def smooth(x, w=20):
    arr = np.array(x, dtype=float)
    if w < 2 or len(arr) < w: return arr
    return np.convolve(arr, np.ones(w)/w, mode='same')

COLORS = {'Q-Learning':'#1565C0', 'SARSA':'#C62828'}

In [ ]:
# ── Figure 1: Learning Curves ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.subplots_adjust(wspace=0.28)

for ax_i, (key, ylabel, title) in enumerate([
    ('train_cd',     'Mean Chamfer Distance (mm)', 'Training Chamfer Distance'),
    ('train_reward', 'Cumulative Reward',           'Training Cumulative Reward')
]):
    ax = axes[ax_i]
    for name, hist, col in [('Q-Learning', ql_hist, COLORS['Q-Learning']),
                              ('SARSA',     sarsa_hist, COLORS['SARSA'])]:
        raw = hist[key]
        ax.plot(raw,         alpha=0.15, color=col)
        ax.plot(smooth(raw, 30), lw=2.2, color=col, label=name)
        if ax_i == 0 and hist['eval_episodes']:
            ax.scatter(hist['eval_episodes'], hist['eval_cd'],
                       marker='D', s=30, color=col, zorder=5)
    ax.set_xlabel('Training Episode', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(True, alpha=0.25)

fig.suptitle('Q-Learning vs SARSA — Training Curves', fontsize=13, fontweight='bold')
plt.savefig('rl_results/fig1_learning_curves.pdf', bbox_inches='tight')
plt.show()
print('✓ Fig 1 saved')

In [ ]:
# ── Figure 2: Evaluation Bar Chart ───────────────────────────────────────
agent_names = list(results.keys())
x      = np.arange(len(EP_NAMES))
width  = 0.8 / len(agent_names)
palette= ['#1565C0','#C62828','#2E7D32','#F57F17','#6A1B9A','#00695C']

fig, ax = plt.subplots(figsize=(11, 4.8))
for i, name in enumerate(agent_names):
    means = [np.mean(results[name][et]) for et in EP_NAMES]
    stds  = [np.std(results[name][et])  for et in EP_NAMES]
    offset = (i - len(agent_names)/2 + 0.5) * width
    ax.bar(x+offset, means, width*0.9, label=name,
           color=palette[i%len(palette)], alpha=0.88, zorder=3)
    ax.errorbar(x+offset, means, yerr=stds,
                fmt='none', color='#333', capsize=3, lw=1.2, zorder=4)

ax.set_xticks(x); ax.set_xticklabels(EP_NAMES, fontsize=11)
ax.set_ylabel('Mean Chamfer Distance (mm)', fontsize=11)
ax.set_title('Evaluation: Chamfer Distance by Episode Type (lower = better)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8, ncol=2); ax.grid(True, axis='y', alpha=0.25)
ax.set_axisbelow(True)
plt.savefig('rl_results/fig2_eval_bar.pdf', bbox_inches='tight')
plt.show()
print('✓ Fig 2 saved')

In [ ]:
# ── Figure 3: k_obs Trajectory per Episode Type ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.0), sharey=True)
fig.subplots_adjust(wspace=0.08)

fixed_base = FixedGainBaseline(48.0, env)

for ci, (ep_name, ep_code) in enumerate(zip(EP_NAMES, EPISODE_TYPES)):
    ax = axes[ci]
    ql_r  = run_episode_ql(env,    ql,    ep_type=ep_code, train=False)
    sa_r  = run_episode_sarsa(env, sarsa, ep_type=ep_code, train=False)
    fx_r  = run_episode_baseline(env, fixed_base, ep_type=ep_code)

    frames = np.arange(EPISODE_LEN)
    ax.plot(frames, ql_r['k_traj'], '#1565C0', lw=2.0, label='Q-Learning')
    ax.plot(frames, sa_r['k_traj'], '#C62828', lw=2.0, label='SARSA', ls='--')
    ax.plot(frames, fx_r['k_traj'], '#999',    lw=1.2, label='Fixed 48', ls=':')
    ax.axhline(48, color='#bbb', lw=0.8, ls=':')

    if ep_code == EP_OCCLUSION:
        ax.axvspan(100, 130, alpha=0.12, color='orange')
        ax.text(115, K_OBS_MAX*0.92, 'occlusion', ha='center', fontsize=8, color='darkorange')
    if ep_code == EP_CONTACT:
        ax.axvline(150, color='red', lw=1.2, ls='--', alpha=0.6)
        ax.text(152, K_OBS_MAX*0.92, 'impulse', fontsize=8, color='red')

    ax.set_title(ep_name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Frame', fontsize=10)
    ax.grid(True, alpha=0.2)

axes[0].set_ylabel('k_obs (N/m)', fontsize=10)
axes[0].legend(fontsize=9, loc='upper left')
axes[0].set_ylim(K_OBS_MIN*0.8, K_OBS_MAX*1.05)

fig.suptitle('Learned k_obs Trajectory by Episode Type', fontsize=11, fontweight='bold')
plt.savefig('rl_results/fig3_kobs_traj.pdf', bbox_inches='tight')
plt.show()
print('✓ Fig 3 saved')

In [ ]:
# ── Figure 4: Q-Table Heatmap ─────────────────────────────────────────────
best_action = np.argmax(ql.Q, axis=1)                   # (625,)
policy      = best_action.reshape([N_STATE_BINS]*4)      # (5,5,5,5)
policy_2d   = policy.max(axis=3).max(axis=1)             # (5,5) dCD × Δcloud

labels_cd    = [f'{v:.0f}' for v in np.linspace(STATE_LOW[0],  STATE_HIGH[0],  N_STATE_BINS)]
labels_cloud = [f'{v:.1f}' for v in np.linspace(STATE_LOW[2],  STATE_HIGH[2],  N_STATE_BINS)]

fig, ax = plt.subplots(figsize=(6, 5))
cmap = plt.cm.get_cmap('RdYlGn_r', 3)
im   = ax.imshow(policy_2d, cmap=cmap, vmin=-0.5, vmax=2.5, aspect='auto', origin='lower')
action_labels = {0:'↓ Decr', 1:'= Hold', 2:'↑ Incr'}
for i in range(N_STATE_BINS):
    for j in range(N_STATE_BINS):
        ax.text(j, i, action_labels[policy_2d[i,j]], ha='center', va='center',
                fontsize=9, fontweight='bold',
                color='white' if policy_2d[i,j] != 1 else 'black')
ax.set_xticks(range(N_STATE_BINS)); ax.set_xticklabels(labels_cloud, fontsize=9)
ax.set_yticks(range(N_STATE_BINS)); ax.set_yticklabels(labels_cd, fontsize=9)
ax.set_xlabel('Δp_cloud (mm)', fontsize=10)
ax.set_ylabel('Chamfer Distance (mm)', fontsize=10)
ax.set_title('Greedy Policy — Q-Learning\n(max over speed & k_prev dims)',
             fontsize=10, fontweight='bold')
cbar = plt.colorbar(im, ax=ax, ticks=[0,1,2], shrink=0.85)
cbar.ax.set_yticklabels(['Decrease','Hold','Increase'], fontsize=9)
plt.savefig('rl_results/fig4_qtable.pdf', bbox_inches='tight')
plt.show()
print('✓ Fig 4 saved')

In [ ]:
# ── Figure 5: ε-Decay Curve ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(ql_hist['epsilon'], '#1565C0', lw=2, label='Q-Learning')
ax.plot(sarsa_hist['epsilon'], '#C62828', lw=2, ls='--', label='SARSA')
ax.set_xlabel('Training Episode', fontsize=11)
ax.set_ylabel('ε (exploration rate)', fontsize=11)
ax.set_title('ε-Greedy Exploration Decay', fontsize=11, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.25)
plt.savefig('rl_results/fig5_epsilon.pdf', bbox_inches='tight')
plt.show()
print('✓ Fig 5 saved')

In [ ]:
# ── Download all results ──────────────────────────────────────────────────
import zipfile, glob
with zipfile.ZipFile('rl_results.zip', 'w') as zf:
    for f in glob.glob('rl_results/*'):
        zf.write(f)

from google.colab import files
files.download('rl_results.zip')
print('✓ Download triggered')